<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> 一书的补充代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第 2 章：使用预训练 LLM 生成文本

本 notebook 中使用的包：

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.13
torch version: 2.10.0
tokenizers version: 0.21.4


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F01_raschka.webp?1" width="500px">

&nbsp;
## 2.1 LLM 文本生成简介

- 本节无代码
- LLM 是如何生成文本的？
- 本章是一个准备章节：设置我们在全书中将使用的编程环境和 LLM
- 我们还会编写文本生成函数，这些函数将在后续章节中使用和扩展

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F02_raschka.webp?1" width="300px">

- LLM（和神经网络）流程图传统上是从上到下阅读和绘制的

&nbsp;
## 2.2 设置编程环境

- 如果你正在阅读本书，你之前很可能已经用 Python 编过程
- 如果你已经设置好了 Python 环境（Python 3.10 或更新版本），安装依赖最简单的方式是使用 `pip`：

In [2]:
#!pip install -r https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/refs/heads/main/requirements.txt

- 本章的依赖也可以手动安装：

In [3]:
#!pip install torch>=2.10.0 tokenizers>=0.22.2 reasoning-from-scratch

- 我偏好使用广受推荐的 [uv](https://docs.astral.sh/uv/) Python 包和项目管理器
- 要安装 `uv`，请从官方网站为你的操作系统运行安装命令：https://docs.astral.sh/uv/getting-started/installation/
- 然后，克隆 GitHub 仓库：

In [4]:
#!git clone --depth 1 https://github.com/rasbt/reasoning-from-scratch.git

- 如果你没有安装 `git`，也可以从 Manning 网站手动下载源代码仓库，或点击此链接：https://github.com/rasbt/reasoning-from-scratch/archive/refs/heads/main.zip（下载后解压）

- 在终端中，导航到 `reasoning-from-scratch` 文件夹
- 运行 `uv run jupyter lab` 启动 JupyterLab，然后打开空白 notebook 或本章的 notebook
- 此命令还会设置本地虚拟环境（通常在 `.venv/` 中），并自动从 `reasoning-from-scratch` 文件夹内的 `pyproject.toml` 文件安装所有依赖

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F03_raschka.webp?1" width="500px">

- 如需更多安装细节和选项，请参阅 [../02_setup-tips/python-instructions.md](../02_setup-tips/python-instructions.md)

&nbsp;
## 2.3 了解硬件需求和建议

- 如果你是 PyTorch 新手，我建议阅读我的 [PyTorch in One Hour: From Tensors to Training Neural Networks on Multiple GPUs](https://sebastianraschka.com/teaching/pytorch-1h/) 教程
- 如果你按照上一节操作，你应该已经安装了 PyTorch
- 手动检查你的 PyTorch 安装是否支持 GPU；查看你的机器支持什么：

In [5]:
import torch


print(f"PyTorch version {torch.__version__}")

if torch.cuda.is_available():
    print(f"CUDA/ROCm GPU: {torch.cuda.get_device_name(0)}")

elif torch.xpu.is_available():
    print(f"Intel GPU: {torch.xpu.get_device_name(0)}")

elif torch.backends.mps.is_available():
    print("Apple Silicon GPU")

else:
    print("Only CPU")

PyTorch version 2.10.0
Apple Silicon GPU


- 根据章节不同，代码会在 NVIDIA (CUDA) GPU 可用时自动使用它，否则在 CPU 上运行（或在特定章节推荐时使用 Apple Silicon GPU）
- 第 2-4 章可以在 CPU 上在合理时间内执行
- 第 5-7 章的代码在 CPU 上执行会非常慢，建议使用支持 CUDA 的 GPU（关于确切的资源需求，详见后续章节）
- 我个人偏好使用 [Lightning AI Studio](https://lightning.ai/)，注册和验证后可提供免费计算额度；另外，[Google Colab](https://colab.research.google.com/) 也是不错的选择

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F04_raschka.webp" width="500px">

- 如需云计算建议，请参阅 [../02_setup-tips/gpu-instructions.md](../02_setup-tips/gpu-instructions.md)
- 但目前还不需要使用 GPU；前几章在非 GPU 硬件上也能正常运行

&nbsp;
## 2.4 为 LLM 准备输入文本

- 在本节中，我们将学习如何使用分词器（tokenizer）；我们用它将输入文本转换（编码）为 token ID 表示，作为 LLM 的输入
- 我们还使用分词器将 LLM 输出转换（解码）回人类可读的文本表示

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F05_raschka.webp?1" width="500px">

- 如前所述，从零实现 LLM 和分词器超出了本书的范围，本书专注于在现有 LLM 和分词器之上从零实现推理方法
- 在本书中，我们将使用一个预训练 LLM，将在下一节加载它；这里我们先加载与之配套的分词器
- 我准备了一个 `reasoning_from_scratch` Python 包，提供基础 LLM 和对应的分词器，我借助 [`tokenizers`](https://github.com/huggingface/tokenizers) Python 库编写了这些代码
- `reasoning_from_scratch` 包的代码是本书补充代码的一部分，根据第 2.2 节的说明，它应该已经安装好了

- 接下来，我们下载分词器文件（这是 Qwen3 基座 LLM 的分词器，下一节会详细介绍）：

In [6]:
from reasoning_from_scratch.qwen3 import download_qwen3_small

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

- 现在，我们可以将分词器文件中的设置加载到 `Qwen3Tokenizer` 中：

In [7]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

- 由于我们还没有加载 LLM 本身，先做一个简单的往返测试：将文本编码为 token ID，然后再解码回字符串表示：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F06_raschka.webp" width="500px">

In [8]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)

In [9]:
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

840 --> Ex
20772 --> plain
3460 -->  large
4128 -->  language
4119 -->  models
13 --> .


In [10]:
text = tokenizer.decode(input_token_ids_list)
print(text)

Explain large language models.


- 对于 `Qwen3Tokenizer`，大约有 15.1 万个唯一 token（词汇表大小）

- 关于分词的更多资源：
  - [Build a Large Language Model (from Scratch)](https://mng.bz/M96o) 第 2 章
  - [Implementing A Byte Pair Encoding (BPE) Tokenizer From Scratch](https://sebastianraschka.com/blog/2025/bpe-from-scratch.html)

&nbsp;
## 2.5 加载预训练模型

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F07_raschka.webp" width="500px">

- 正如上一节加载分词器时所暗示的，本书使用 Qwen3 0.6B；经过深思熟虑选择使用哪个开放权重基座模型后，我选择了 Qwen3，因为
  - 截至撰写本书时，Qwen3 在建模性能方面是领先的开放权重模型
  - Qwen3 0.6B 比 Llama 3 1B 更节省内存
  - 既有基座模型（我们用于推理模型开发）也有官方推理变体，可作为参考模型
- （注意，规范拼写中“Qwen3”不含空格，而“Llama 3”含空格）
- 秉承“从零开始”的精神，我们使用的是我用纯 PyTorch 重新实现的 Qwen3，不依赖任何外部 LLM 库；这个从零实现与原始 Qwen3 模型权重兼容
- 但是，本书不会逐一讲解 Qwen3 的代码实现，因为这本身就可以写一整本书（类似于我的 [Build A Large Language Model (From Scratch)](https://github.com/rasbt/LLMs-from-scratch) 一书）；相反，本书（Build A Reasoning Model From Scratch）专注于在基座模型（这里是 Qwen3）之上从零实现推理方法
- Qwen3 模型代码详见附录 C
- 加载推理变体和更大的 Qwen3 模型详见附录 D
- 更多细节请参阅 Qwen3 的 [GitHub 仓库](https://github.com/QwenLM/Qwen3) 和[技术报告](https://arxiv.org/abs/2505.09388)

- 该模型特意设计得较小（但仍然非常强大），以便在消费级硬件上运行
- 它可以在 CPU、NVIDIA GPU (`"cuda"`)、Apple Silicon GPU (`"mps"`) 和 Intel GPU (`"xpu"`) 上正常运行；关于性能权衡的更多内容将在本章后面讨论

In [11]:
def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")
        
        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 9):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")

    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU")

    else:
        device = torch.device("cpu")
        print("Using CPU")

    return device

device = get_device()

Using Apple Silicon GPU (MPS)


- 建议在首次运行时使用 `"cpu"` 运行代码，因此我们在下面硬编码设备：

In [12]:
# 推荐：首次运行时使用 CPU
device = torch.device("cpu")

- 然后，我们下载包含预训练模型权重的文件，大小约 1.5 GB：

In [13]:
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

✓ qwen3/qwen3-0.6B-base.pth already up-to-date


- 下面展示了我们正在加载的 Qwen3 0.6B 模型的架构结构，供熟悉 LLM 架构的读者参考，但请注意，对于本书来说，理解此架构**并非**必要或重要，因为我们在后续章节中不会修改它，而是在其上添加推理技术

- 我为 [reasoning-from-scratch](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3.py) Python 包从零编写了 Qwen3 模型架构；源代码也展示在附录 C 中；但再次强调，这只是作为对好奇读者的额外内容，理解这些内部细节对于跟随本书其余部分并非必要

In [14]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"

model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))

model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F08_raschka.webp" width="300px">

&nbsp;
## 2.6 理解 LLM 顺序文本生成过程

- 在本节中，我们编写一个简单的包装函数，以便使用 LLM 生成文本（我们将在第 4 章中为此函数添加额外功能）

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F09_raschka.webp?1" width="500px">

- LLM 每次生成一个词：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F10_raschka.webp?2" width="500px">

- 上图是简化版本，只显示了新生成的词；下图放大了第一次迭代的细节：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp" width="3b00px">

In [15]:
example = torch.tensor([1, 2, 3]) 
print(example)
print(example.unsqueeze(0))

tensor([1, 2, 3])
tensor([[1, 2, 3]])


In [16]:
example = torch.tensor([[1, 2, 3]]) 
print(example)
print(example.squeeze(0))

tensor([[1, 2, 3]])
tensor([1, 2, 3])


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp?2" width="300px">

In [17]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_token_ids_list)}")

input_tensor = torch.tensor(input_token_ids_list)
input_tensor_fmt = input_tensor.unsqueeze(0).to(device)

with torch.inference_mode():
    output_tensor = model(input_tensor_fmt)

output_tensor_fmt = output_tensor.squeeze(0)
print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")

Number of input tokens: 6
Formatted Output tensor shape: torch.Size([6, 151936])


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F12_raschka.webp" width="500px">

In [18]:
last_token = output_tensor_fmt[-1]
print(last_token)

tensor([ 7.3750,  2.0312,  8.0000,  ..., -2.5469, -2.5469, -2.5469],
       dtype=torch.bfloat16)


In [19]:
print(torch.argmax(last_token, dim=-1, keepdim=True))

tensor([20286])


In [20]:
print(tokenizer.decode([20286]))

 Large


In [21]:
example = torch.tensor([-2, 1, 3, 1])
print(torch.max(example))
print(torch.argmax(example))

tensor(3)
tensor(2)


&nbsp;
## 2.7 编写最小文本生成函数


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F13_raschka.webp" width="500px">

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F14_raschka.webp?2" width="500px">

- `generate_text_basic_stream` 函数实现了这个顺序文本生成过程：

In [22]:
@torch.inference_mode()
def generate_text_basic_stream(
    model,
    token_ids,
    max_new_tokens, 
    eos_token_id=None
):
    model.eval()

    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        # 如果遇到序列结束 token 则停止
        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token  # Yield each token as it's generated
        
        token_ids = torch.cat([token_ids, next_token], dim=1)

- 让我们用它对一个简单的 `"Explain large language models in a single sentence."` 提示生成 100 个 token 的响应，看看它是如何工作的（推理部分将在后续章节中介绍）
- 以下代码会比较慢，根据你的计算机可能需要 1-3 分钟完成（我们将在后面的章节中加速它）

In [23]:
prompt = "Explain large language models in a single sentence."
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)
max_new_tokens = 100


for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True  # 禁用缓冲以便 token 实时打印
    )

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.<|endoftext|>Human language is a complex and dynamic system that has evolved over millions of years to enable effective communication and social interaction. It is composed of a vast array of symbols, including letters, numbers, and words, which are used to convey meaning and express thoughts and ideas. The evolution of language has

- 注意 LLM 很好地遵循了指令，但响应在 `<|endoftext|>` 之后变得无意义/离题了，该 token 是训练期间用作不同文档之间分隔符的
- 在使用 LLM 时，我们希望它在遇到此 token 后停止生成

In [24]:
print(tokenizer.encode("<|endoftext|>"))

[151643]


- 为了方便，此 token ID 存储为分词器属性（eos = end of sequence，序列结束）：

In [25]:
print(tokenizer.eos_token_id)

151643


- 我们可以用它来告诉 LLM（或者更准确地说是 `generate_text_basic_stream` 函数）何时停止生成文本

In [26]:
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id  # 使用 EOS token
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

- 以上响应是在 CPU 上运行代码时得到的结果，生成的文本可能因设备不同而略有差异

- 在我们结束本节并了解如何加速代码之前，让我们实现一个简单的基准测试函数来跟踪计算性能

In [27]:
import warnings

def generate_stats(output_token_ids, tokenizer, start_time,
                   end_time):
    total_time = end_time - start_time
    print(f"\n\nTime: {total_time:.2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} tokens/sec")

    for name, backend in (("CUDA", getattr(torch, "cuda", None)),
                          ("XPU", getattr(torch, "xpu", None))):
        if backend is not None and backend.is_available():

            # 检查我们是否实际在使用此后端
            device_type = output_token_ids.device.type
            if device_type != name.lower():
                warnings.warn(
                    f"{name} is available but tensors are on "
                    f"{device_type}. Memory stats may be 0."
                )
    
            # 如果支持则同步（对异步后端很重要）
            if hasattr(backend, "synchronize"):
                backend.synchronize()
            
            max_mem_bytes = backend.max_memory_allocated()
            max_mem_gb = max_mem_bytes / (1024 ** 3)
            print(f"Max {name} memory allocated: {max_mem_gb:.2f} GB")
            backend.reset_peak_memory_stats()

In [28]:
import time

start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # 收集生成的 token

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Time: 1.39 sec
29 tokens/sec


&nbsp;
## 2.8 通过 KV 缓存加速推理

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F15_raschka.webp?2" width="500px">

- 请注意，本书中的代码强调代码可读性，关于优化可以单独写一整本书
- 这里，我们看一个叫做“KV 缓存”（KV caching）的工程技巧（KV 指的是 LLM 注意力机制（attention mechanism）中的键（keys）和值（values））
- 如果你不熟悉这些术语，不用担心，你只需要知道有一种方法可以存储（缓存）在每次迭代中重复使用的中间值

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F16_raschka.webp" width="500px">

- 关于 KV 缓存机制的更多细节，请参阅我的 [Understanding and Coding the KV Cache in LLMs from Scratch](https://magazine.sebastianraschka.com/p/coding-the-kv-cache-in-llms) 文章
- 下面是使用 KV 缓存的 `generate_text_basic_stream` 函数的修改版本

In [29]:
from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()
def generate_text_basic_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])  # 新增
    model.reset_kv_cache()                           # 新增

    out = model(token_ids, cache=cache)[:, -1]
    for _ in range(max_new_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token
        out = model(next_token, cache=cache)[:, -1]

- 使用方式与之前类似：

In [30]:
start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # 收集生成的 token

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Time: 0.84 sec
49 tokens/sec


- 可以看到，它比之前快了数量级（28 tokens/sec 而非 4 tokens/sec；在 Mac Mini M4 CPU 上运行）

&nbsp;
## 2.9 通过 PyTorch 模型编译加速推理

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F17_raschka.webp?2" width="500px">

- 另一种大幅加速模型推理（文本生成）的技术是使用 `torch.compile`
- 用法很简单，只需对模型调用 `torch.compile`（更多选项请参阅[文档](https://docs.pytorch.org/docs/stable/torch.compiler_api.html)）

In [31]:
major, minor = map(int, torch.__version__.split(".")[:2])
if (major, minor) >= (2, 8):
    # 这避免了重新触发模型编译
    # 在 PyTorch 2.8 及更新版本中
    # 如果模型包含类似 self.pos = self.pos + 1 的代码
    torch._dynamo.config.allow_unspec_int_on_nn_module = True

model_compiled = torch.compile(model)

# 如果你在 "mps" 设备上使用 torch.compile 遇到问题并得到 InductorError，
# 请确保你使用的是 PyTorch 2.9 或更新版本

---

**Windows 注意事项 1**

- 编译在 Windows 上可能比较棘手
- `torch.compile()` 使用 Inductor，它会 JIT 编译内核，需要一个可用的 C/C++ 工具链
- 对于 CUDA，Inductor 还依赖 Triton，可通过社区包 `triton-windows` 获取
  - 如果你看到 `cl not found`，请[安装带有“C++ 工作负载”的 Visual Studio Build Tools](https://learn.microsoft.com/en-us/cpp/build/vscpp-step-0-installation?view=msvc-170) 并从“x64 Native Tools”提示符运行 Python
  - 如果你在 CUDA 下看到 `triton not found`，请安装 `triton-windows`（例如 `uv pip install "triton-windows<3.4"`）
- 对于 CPU，一位读者进一步建议参考此 [PyTorch Inductor guide for Windows](https://docs.pytorch.org/tutorials/unstable/inductor_windows.html)
  - 这里很重要的是在安装 Visual Studio 2022 时安装英语语言包，以避免 UTF-8 错误
  - 另外请注意，代码需要通过“Visual Studio 2022 Developer Command Prompt”而非 notebook 运行
- 如果此设置过于复杂，你可以跳过编译；**编译是可选的，所有代码示例不编译也能正常工作**

**Windows 注意事项 2**

- 读者反馈在 Windows 上使用默认设置运行 `torch.compile` 没有速度提升；但是使用 `"max-autotune"` 模式运行 `torch.compile` 可以获得 2 倍加速：`torch.compile(model, mode="max-autotune")`

---

- 第一次迭代可能较慢，因为需要进行初始编译和优化；因此我们多次重复文本生成
- 首先，从非缓存版本开始（这可能比较慢，可能需要几分钟）

In [32]:
for i in range(3):

    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # 收集生成的 token
    
    end_time = time.time()
    

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

    print(f"\n{30*'-'}\n")

W0213 17:02:09.090000 73246 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays.

Warm-up run


Time: 27.15 sec
1 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays.

Timed run 1:


Time: 0.82 sec
42 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays.

Timed run 2:


Time: 0.82 sec
42 tokens/sec

------------------------------



- 如上所示，5 tokens/sec 的速度仅比之前略快（4 tokens/sec）
- 现在让我们看看 KV 缓存版本的表现

In [33]:
for i in range(3):
    
    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream_cache(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # 收集生成的 token
    
    end_time = time.time()

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(
        output_token_ids_tensor, tokenizer, start_time, end_time
    )

    print(f"\n{30*'-'}\n")

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Warm-up run


Time: 45.89 sec
0 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Timed run 1:


Time: 0.48 sec
84 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Timed run 2:


Time: 0.45 sec
90 tokens/sec

------------------------------



- 可以看到，编译带来了显著的 2 倍加速（64 tokens/sec 对比 30 tokens/sec）
- 下面是包含更多结果的表格

| 模型      | 模式              | 硬件             | Tokens/sec    | GPU 显存 (VRAM) |
|------------|-------------------|----------------------|---------------|-------------------|
| Qwen3Model | 常规           | Mac Mini M4 CPU      | 5             | -                 |
| Qwen3Model | 常规 + 编译  | Mac Mini M4 CPU      | 5             | -                 |
| Qwen3Model | KV 缓存          | Mac Mini M4 CPU      | 29            | -                 |
| Qwen3Model | KV 缓存 + 编译 | Mac Mini M4 CPU      | 68            | -                 |
|            |                   |                      |               |                   |
| Qwen3Model | 常规           | Mac Mini M4 GPU      | 27            | -                 |
| Qwen3Model | 常规 + 编译  | Mac Mini M4 GPU      | 43            | -                 |
| Qwen3Model | KV 缓存          | Mac Mini M4 GPU      | 41            | -                 |
| Qwen3Model | KV 缓存 + 编译 | Mac Mini M4 GPU      | 71            | -                 |
|            |                   |                      |               |                   |
| Qwen3Model | 常规           | NVIDIA H100 GPU      | 51            | 1.55 GB           |
| Qwen3Model | 常规 + 编译  | NVIDIA H100 GPU      | 164           | 1.81 GB           |
| Qwen3Model | KV 缓存          | NVIDIA H100 GPU      | 48            | 1.52 GB           |
| Qwen3Model | KV 缓存 + 编译 | NVIDIA H100 GPU      | 141           | 1.81 GB           |
|            |                   |                      |               |                   |
| Qwen3Model | 常规           | NVIDIA DGX Spark GPU | 74            | 1.53 GB           |
| Qwen3Model | 常规 + 编译  | NVIDIA DGX Spark GPU | 103           | 1.49 GB           |
| Qwen3Model | KV 缓存          | NVIDIA DGX Spark GPU | 68            | 1.47 GB           |
| Qwen3Model | KV 缓存 + 编译 | NVIDIA DGX Spark GPU | 98            | 1.47 GB           |

- 上面的 NVIDIA DGX Spark 使用 GB10 (Blackwell) GPU
- 注意我们所有示例都使用单个提示运行（即 batch size 为 1）；如果你对批量推理感兴趣，请参阅附录 E

&nbsp;
## 总结

- 本节无代码